In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import requests
url = "https://api.openalex.org/works"
params = {
    "search": "transformer",
    "per-page": 5,
    "mailto": "YOUR_EMAIL",
}
r = requests.get(url, params=params)
print("Status:", r.status_code)
data = r.json()
print("Meta:", data["meta"])
print("Number of results:", len(data["results"]))

Status: 200
Meta: {'count': 1314537, 'db_response_time_ms': 54, 'page': 1, 'per_page': 5, 'groups_count': None, 'x_query': {'oql': 'get works where full text has (transformer)', 'oqo': {'get_rows': 'works', 'filter_rows': [{'column_id': 'fulltext.search', 'value': 'transformer', 'operator': 'has'}]}, 'url': '/works?filter=fulltext.search:transformer&per_page=5'}, 'cost_usd': 0.001}
Number of results: 5


In [3]:
paper=data["results"][0]
print(paper.keys())

dict_keys(['id', 'doi', 'title', 'display_name', 'relevance_score', 'publication_year', 'publication_date', 'ids', 'language', 'primary_location', 'volume', 'type', 'indexed_in', 'open_access', 'authorships', 'institutions', 'countries_distinct_count', 'institutions_distinct_count', 'corresponding_author_ids', 'corresponding_institution_ids', 'apc_list', 'apc_paid', 'fwci', 'has_fulltext', 'cited_by_count', 'citation_normalized_percentile', 'cited_by_percentile_year', 'biblio', 'is_retracted', 'is_paratext', 'is_xpac', 'primary_topic', 'topics', 'keywords', 'concepts', 'mesh', 'locations_count', 'locations', 'best_oa_location', 'sustainable_development_goals', 'sustainable_development_goals_aurora', 'study_designs', 'awards', 'funders', 'has_content', 'content_urls', 'referenced_works_count', 'referenced_works', 'related_works', 'abstract_inverted_index', 'counts_by_year', 'updated_date', 'created_date'])


In [13]:
for paper in data["results"]: 
    title=paper["title"][:50]
    total=paper["cited_by_count"]
    yearly=paper["counts_by_year"]
    yearly_sum=sum(entry["cited_by_count"] for entry in yearly)
    years = [entry["year"]for entry in yearly]          

    print(title)
    print("  published:", paper["publication_year"])
    print("  total:", total, "| sum of yearly:", yearly_sum)
    print("  years covered:", sorted(years))               
    print()
    

Transformer in Transformer
  published: 2021
  total: 1012 | sum of yearly: 1012
  years covered: [2021, 2022, 2023, 2024, 2025, 2026]

A survey of transformers
  published: 2022
  total: 1255 | sum of yearly: 1255
  years covered: [2020, 2021, 2022, 2023, 2024, 2025, 2026]

Swin Transformer: Hierarchical Vision Transformer 
  published: 2021
  total: 32986 | sum of yearly: 32984
  years covered: [2012, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

Point Transformer
  published: 2021
  total: 2466 | sum of yearly: 2466
  years covered: [2021, 2022, 2023, 2024, 2025, 2026]

A survey of the vision transformers and their CNN-
  published: 2023
  total: 452 | sum of yearly: 452
  years covered: [2023, 2024, 2025, 2026]



## Design decisions
- Citation label: citations in pub year + next 2 years (from counts_by_year)
- Only papers published 2015–2023, so every label window is complete
- Citations dated before pub year are ignored (arXiv preprint effect)
- Never use as features (leakage): cited_by_count, fwci,
  citation_normalized_percentile, cited_by_percentile_year, related_works

In [32]:
def reconstruct_abstract(abstract):
    if not abstract:
        return None

    words = []

    for word, positions in abstract.items():
        for position in positions:
            words.append((position, word))

    words.sort()

    return " ".join(word for position, word in words)


# Call the function


NO ABSTRACT


In [47]:

for paper in data["results"]:
     abstract_text = reconstruct_abstract(paper["abstract_inverted_index"])
     if abstract_text:
            print(abstract_text[:200])
     else:
            print("NO ABSTRACT")

# Check before slicing
        

Transformer is a new kind of neural architecture which encodes the input data as powerful features via the attention mechanism. Basically, the visual transformers first divide the input images into se
Transformers have achieved great success in many artificial intelligence fields, such as natural language processing, computer vision, and audio processing. Therefore, it is natural to attract lots of
This paper presents a new vision Transformer, called Swin Transformer, that capably serves as a general-purpose backbone for computer vision. Challenges in adapting Transformer from language to vision
Self-attention networks have revolutionized natural language processing and are making impressive strides in image analysis tasks such as image classification and object detection. Inspired by this su
NO ABSTRACT


In [55]:
def extract_record(paper):
    # Handle topic information safely
    primary_topic = paper.get("primary_topic")

    if primary_topic:
        topic = primary_topic.get("display_name")

        subfield_data = primary_topic.get("subfield")
        subfield = subfield_data.get("display_name") if subfield_data else None

        field_data = primary_topic.get("field")
        field = field_data.get("display_name") if field_data else None
    else:
        topic = None
        subfield = None
        field = None

    # Reconstruct abstract
    abstract_text = reconstruct_abstract(  paper.get("abstract_inverted_index"))

    # Return cleaned record
    return {
        "id": paper.get("id"),
        "title": paper.get("title"),
        "abstract": abstract_text,
        "publication_date": paper.get("publication_date"),
        "publication_year": paper.get("publication_year"),
        "type": paper.get("type"),
        "language": paper.get("language"),
        "referenced_works": paper.get("referenced_works"),
        "counts_by_year": paper.get("counts_by_year"),
        "topic": topic,
        "subfield": subfield,
        "field": field
    }

In [56]:
for paper in data["results"]:
    rec = extract_record(paper)

    print(
        rec["title"], "|",
        rec["field"], "|",
        rec["subfield"], "|",
        rec["publication_year"]
    )

Transformer in Transformer | Computer Science | Computer Vision and Pattern Recognition | 2021
A survey of transformers | Computer Science | Artificial Intelligence | 2022
Swin Transformer: Hierarchical Vision Transformer using Shifted Windows | Computer Science | Computer Vision and Pattern Recognition | 2021
Point Transformer | Engineering | Computational Mechanics | 2021
A survey of the vision transformers and their CNN-transformer based variants | Computer Science | Computer Vision and Pattern Recognition | 2023


In [57]:

import json

# Step 1: Extract and clean all papers
cleaned_records = [
    extract_record(paper)
    for paper in data["results"]
]

# Step 2: Check how many papers have abstracts
n_with_abstract = sum(
    1 for record in cleaned_records if record["abstract"]
)

print(f"Abstracts available: {n_with_abstract}/{len(cleaned_records)}")

# Step 3: Save cleaned records to a JSON file
with open("cleaned_papers.json", "w", encoding="utf-8") as file:
    json.dump(cleaned_records, file, indent=4, ensure_ascii=False)

print(f"Saved {len(cleaned_records)} papers to cleaned_papers.json")


Abstracts available: 4/5
Saved 5 papers to cleaned_papers.json


In [61]:
import requests, json, time

url = "https://api.openalex.org/works"
params = {
    "filter": "primary_topic.field.id:17,publication_year:2020,language:en",
    "per-page": 200,              # max allowed is 200
    "cursor": "*",                # "*" = start from the beginning
    "mailto": "YOUR_EMAIL",
}

r = requests.get(url, params=params)
data = r.json()

records = [extract_record(p) for p in data["results"]]

with open("papers.jsonl", "a", encoding="utf-8") as f:     # append mode
    for rec in records:
        f.write(json.dumps(rec,ensure_ascii=False) + "\n")                                # one record as one line

print("Saved", len(records))
print("Next cursor:", data["meta"]["next_cursor"])

Saved 200
Next cursor: IlsxMDAuMCwgMTA2NiwgJ2h0dHBzOi8vb3BlbmFsZXgub3JnL1cyOTg4OTM3ODA0J10i
